# Experiment 06 — Lightweight Perch Sequence Head

**Builds on:** Exp05's Perch scores with max genus proxies and site/hour priors.

**New in this experiment:** train a compact GRU sequence head over 5-second Perch windows grouped by soundscape file. This is a deliberately small ProtoSSM-style test: learn temporal context from Perch embeddings/logits without the full two-pass discussion stack.

**Hypothesis:** A sequence head can capture within-file persistence and co-occurrence patterns missed by independent LR probes, and may blend with Exp05 if it adds independent signal.

In [1]:
import json
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.special import expit as sigmoid
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(42)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EMB_CACHE = CACHE_DIR / "exp02_embeddings.npy"
LOG_CACHE = CACHE_DIR / "exp02_logits.npy"
EXP05_RESULTS = CACHE_DIR / "exp05_priors_proxies_results.csv"
RESULTS_CSV = CACHE_DIR / "exp06_sequence_head_results.csv"
BEST_JSON = CACHE_DIR / "exp06_sequence_head_best.json"

N_SPLITS = 5
MAX_WINDOWS = 12
EMB_DIM = 1536
SCORE_DIM = 234
HIDDEN = 96
EPOCHS = 35
PATIENCE = 6
BATCH_SIZE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
POS_WEIGHT_CAP = 20.0
BLEND_GRID = [0.10, 0.20, 0.30, 0.40, 0.50, 0.70]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Device:", device)

Project root: /Users/jroessler/Development/kaggle/birdclef
Device: cpu


In [2]:
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
taxon_ids = taxonomy["primary_label"].tolist()
taxon_to_idx = {tid: i for i, tid in enumerate(taxon_ids)}
N_CLASSES = len(taxon_ids)

perch_labels = pd.read_csv(DATA_DIR / "models/perch_tf/assets/labels.csv", header=0)
perch_labels.columns = ["scientific_name"]
perch_labels["bc_index"] = np.arange(len(perch_labels))
mapping = taxonomy.merge(perch_labels, on="scientific_name", how="left")
comp_to_bc = {
    taxon_to_idx[str(row["primary_label"])] : int(row["bc_index"])
    for _, row in mapping[mapping["bc_index"].notna()].iterrows()
}

def time_to_sec(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

raw_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
raw_df["start_sec"] = raw_df["start"].apply(time_to_sec)
labels_df = (
    raw_df
    .drop_duplicates(subset=["filename", "start_sec"])
    .sort_values(["filename", "start_sec"])
    .reset_index(drop=True)
)

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(";"):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

Y = np.stack(labels_df["primary_label"].apply(parse_labels).values)
emb_full = np.load(EMB_CACHE).astype(np.float32)
logits_full = np.load(LOG_CACHE).astype(np.float32)

def build_scores_with_max_proxies(logits):
    scores = np.zeros((len(logits), N_CLASSES), dtype=np.float32)
    for comp_idx, bc_idx in comp_to_bc.items():
        scores[:, comp_idx] = logits[:, bc_idx]
    for _, row in mapping[mapping["bc_index"].isna()].iterrows():
        scientific_name = str(row["scientific_name"])
        genus = scientific_name.split()[0]
        hits = perch_labels[perch_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)]
        if hits.empty:
            continue
        scores[:, taxon_to_idx[str(row["primary_label"])]] = logits[:, hits["bc_index"].astype(int).values].max(axis=1)
    return scores

scores_full = build_scores_with_max_proxies(logits_full)
prob_scores = sigmoid(scores_full).astype(np.float32)

files = labels_df["filename"].drop_duplicates().tolist()
file_to_rows = {fname: grp.index.to_numpy() for fname, grp in labels_df.groupby("filename", sort=False)}
file_groups = np.array(files)
file_folds = list(GroupKFold(n_splits=N_SPLITS).split(files, groups=files))

print(f"Rows: {len(labels_df)}, files: {len(files)}, active classes: {(Y.sum(axis=0) > 0).sum()}")
print("Per-file window count range:", labels_df.groupby("filename").size().min(), labels_df.groupby("filename").size().max())

Rows: 739, files: 66, active classes: 75
Per-file window count range: 2 12


In [3]:
def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))

def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        positives = y_true[:, c].sum()
        if 0 < positives < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return float(np.mean(aucs)), len(aucs)

def evaluate(name, pred, extra=None):
    auc, n = macro_auc(Y, pred)
    row = {"method": name, "macro_auc": auc, "auc_classes": n, "padded_cmap": padded_cmap(Y, pred)}
    if extra:
        row.update(extra)
    return row

class SequenceDataset(Dataset):
    def __init__(self, file_names, emb, probs, labels):
        self.file_names = list(file_names)
        self.emb = emb
        self.probs = probs
        self.labels = labels

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        fname = self.file_names[idx]
        rows = file_to_rows[fname]
        x_emb = self.emb[rows]
        x_prob = self.probs[rows]
        y = self.labels[rows]
        mask = np.zeros(MAX_WINDOWS, dtype=np.float32)
        feat = np.zeros((MAX_WINDOWS, EMB_DIM + SCORE_DIM), dtype=np.float32)
        target = np.zeros((MAX_WINDOWS, N_CLASSES), dtype=np.float32)
        n = min(len(rows), MAX_WINDOWS)
        feat[:n] = np.concatenate([x_emb[:n], x_prob[:n]], axis=1)
        target[:n] = y[:n]
        mask[:n] = 1.0
        return torch.from_numpy(feat), torch.from_numpy(target), torch.from_numpy(mask), fname

class GRUSequenceHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_classes):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.15),
        )
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Dropout(0.15),
            nn.Linear(hidden_dim * 2, n_classes),
        )

    def forward(self, x):
        h = self.proj(x)
        h, _ = self.gru(h)
        return self.head(h)

def train_one_fold(train_files, val_files, fold):
    scaler = StandardScaler()
    train_rows = np.concatenate([file_to_rows[f] for f in train_files])
    scaler.fit(emb_full[train_rows])
    emb_scaled = scaler.transform(emb_full).astype(np.float32)

    train_ds = SequenceDataset(train_files, emb_scaled, prob_scores, Y)
    val_ds = SequenceDataset(val_files, emb_scaled, prob_scores, Y)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    pos = Y[train_rows].sum(axis=0)
    neg = len(train_rows) - pos
    pos_weight = np.clip(neg / np.maximum(pos, 1), 1.0, POS_WEIGHT_CAP).astype(np.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device), reduction="none")
    model = GRUSequenceHead(EMB_DIM + SCORE_DIM, HIDDEN, N_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_state, best_val, wait = None, float("inf"), 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for feat, target, mask, _ in train_dl:
            feat = feat.to(device)
            target = target.to(device)
            mask = mask.to(device)
            optimizer.zero_grad()
            logits = model(feat)
            loss = criterion(logits, target)
            loss = (loss * mask.unsqueeze(-1)).sum() / (mask.sum() * N_CLASSES).clamp_min(1.0)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            optimizer.step()

        model.eval()
        losses = []
        with torch.no_grad():
            for feat, target, mask, _ in val_dl:
                feat = feat.to(device)
                target = target.to(device)
                mask = mask.to(device)
                logits = model(feat)
                loss = criterion(logits, target)
                loss = (loss * mask.unsqueeze(-1)).sum() / (mask.sum() * N_CLASSES).clamp_min(1.0)
                losses.append(float(loss.cpu()))
        val_loss = float(np.mean(losses))
        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    pred_rows, pred_vals = [], []
    with torch.no_grad():
        for feat, _, mask, names in val_dl:
            probs = torch.sigmoid(model(feat.to(device))).cpu().numpy()
            mask_np = mask.numpy()
            for i, fname in enumerate(names):
                rows = file_to_rows[fname]
                n = min(len(rows), MAX_WINDOWS)
                pred_rows.extend(rows[:n].tolist())
                pred_vals.append(probs[i, :n])
    return np.array(pred_rows), np.concatenate(pred_vals, axis=0), best_val

In [4]:
oof_seq = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
fold_losses = []

for fold, (tr_file_idx, va_file_idx) in enumerate(file_folds):
    train_files = [files[i] for i in tr_file_idx]
    val_files = [files[i] for i in va_file_idx]
    rows, preds, val_loss = train_one_fold(train_files, val_files, fold)
    oof_seq[rows] = preds
    fold_losses.append(val_loss)
    auc_f, n_f = macro_auc(Y[rows], preds)
    print(f"Fold {fold}: val_loss={val_loss:.5f}, AUC={auc_f:.4f} ({n_f} classes), rows={len(rows)}")

seq_row = evaluate("gru_sequence_head", oof_seq, {"blend_seq": 1.0, "mean_val_loss": float(np.mean(fold_losses))})
print(seq_row)

Fold 0: val_loss=0.24775, AUC=0.8954 (37 classes), rows=161


Fold 1: val_loss=0.23475, AUC=0.8205 (43 classes), rows=152


Fold 2: val_loss=0.42159, AUC=0.7357 (42 classes), rows=143


Fold 3: val_loss=0.23238, AUC=0.9167 (34 classes), rows=137


Fold 4: val_loss=0.18440, AUC=0.8471 (28 classes), rows=146


{'method': 'gru_sequence_head', 'macro_auc': 0.6043175121190298, 'auc_classes': 75, 'padded_cmap': 0.8594917306828305, 'blend_seq': 1.0, 'mean_val_loss': 0.2641746044158936}


In [5]:
# Rebuild the exp05 OOF predictions by reusing the exp05 notebook as the benchmark source.
# This keeps comparison exact without duplicating another full LR/probe implementation here.
exp05_best = pd.read_csv(EXP05_RESULTS).iloc[0].to_dict()
exp05_auc = float(exp05_best["macro_auc"])
exp05_cmap = float(exp05_best["padded_cmap"])
print("Exp05 best from results CSV:", exp05_best)

# Approximate exp05 OOF predictions by rerunning the saved exp05 notebook code is too heavy for this comparison cell,
# so for blend testing we compare the sequence head with the strongest readily available independent baseline:
# max-proxy raw Perch probabilities. If the sequence head cannot beat raw Perch, it is not worth blending with exp05.
raw_perch = prob_scores
rows = [
    evaluate("raw_perch_max_proxy", raw_perch, {"blend_seq": 0.0}),
    seq_row,
]
for alpha in BLEND_GRID:
    blend = alpha * oof_seq + (1.0 - alpha) * raw_perch
    rows.append(evaluate("raw_perch_gru_blend", blend, {"blend_seq": alpha}))

results = pd.DataFrame(rows).sort_values(["padded_cmap", "macro_auc"], ascending=False).reset_index(drop=True)
results["exp05_macro_auc"] = exp05_auc
results["exp05_padded_cmap"] = exp05_cmap
results["beats_exp05_cmap"] = results["padded_cmap"] > exp05_cmap
results.to_csv(RESULTS_CSV, index=False)

best = results.iloc[0].to_dict()
with open(BEST_JSON, "w") as f:
    json.dump(best, f, indent=2)

print("\nTop exp06 configurations")
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nSaved:", RESULTS_CSV)
print("Saved:", BEST_JSON)

Exp05 best from results CSV: {'method': 'tuned_probe_prior_blend', 'macro_auc': 0.9026169150237296, 'auc_classes': 75, 'padded_cmap': 0.904973253044536, 'proxy_reduce': 'max', 'prior_blend': 0.1, 'fitted_probes': 295, 'proxy_targets': 6}



Top exp06 configurations
             method  macro_auc  auc_classes  padded_cmap  blend_seq  mean_val_loss  exp05_macro_auc  exp05_padded_cmap  beats_exp05_cmap
raw_perch_gru_blend     0.8030           75       0.8875     0.1000            NaN           0.9026             0.9050             False
raw_perch_gru_blend     0.8007           75       0.8845     0.2000            NaN           0.9026             0.9050             False
raw_perch_gru_blend     0.7966           75       0.8820     0.3000            NaN           0.9026             0.9050             False
raw_perch_gru_blend     0.7909           75       0.8794     0.4000            NaN           0.9026             0.9050             False
raw_perch_gru_blend     0.7835           75       0.8770     0.5000            NaN           0.9026             0.9050             False
raw_perch_gru_blend     0.7556           75       0.8717     0.7000            NaN           0.9026             0.9050             False
raw_perch_max_p

## Result Notes

Record whether the lightweight sequence head beats `exp05` (`0.9026` AUC, `0.9050` padded cmAP). If not, keep `exp05` as the current submission candidate and treat full ProtoSSM as a larger future experiment.